# Kaggle House Prices Regression

Predict house prices using the California Housing dataset as a proxy for the classic Kaggle competition.

**Approach:**
1. Load California Housing data from sklearn
2. Exploratory Data Analysis: distributions, correlations, geographic patterns
3. Feature Engineering: polynomial features, log-transform target
4. Compare models: Linear Regression, Ridge, Lasso, Random Forest, Gradient Boosting
5. Evaluate: RMSE, R-squared, residual analysis

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_squared_error, r2_score
import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)
sns.set_style('whitegrid')
print('Libraries loaded successfully.')

## 1. Load Data

In [ ]:
housing = fetch_california_housing(as_frame=True)
df = housing.frame.copy()

# Convert target to more interpretable units (100k USD)
print(f'Dataset shape: {df.shape}')
print(f'\nFeatures: {list(housing.feature_names)}')
print(f'Target: MedHouseVal (median house value in $100k)')
print(f'\nPrice range: ${df["MedHouseVal"].min() * 100_000:,.0f} - ${df["MedHouseVal"].max() * 100_000:,.0f}')
df.head()

In [ ]:
df.describe()

## 2. Exploratory Data Analysis

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(18, 9))

for idx, col in enumerate(df.columns):
    ax = axes[idx // 4, idx % 4]
    ax.hist(df[col], bins=40, color='#3498db', edgecolor='white', alpha=0.8)
    ax.set_title(col, fontsize=12)
    ax.set_ylabel('Count')

# Use last subplot for target
axes[1, 3].hist(df['MedHouseVal'], bins=40, color='#e74c3c', edgecolor='white', alpha=0.8)
axes[1, 3].set_title('MedHouseVal (Target)', fontsize=12)
axes[1, 3].axvline(df['MedHouseVal'].median(), color='black', linestyle='--', label=f'Median: {df["MedHouseVal"].median():.2f}')
axes[1, 3].legend()

plt.tight_layout()
plt.show()

In [ ]:
# Correlation heatmap
plt.figure(figsize=(10, 8))
corr = df.corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='RdBu_r', center=0,
            square=True, linewidths=0.5)
plt.title('Feature Correlation Matrix', fontsize=14)
plt.tight_layout()
plt.show()

print('\nCorrelation with MedHouseVal:')
print(corr['MedHouseVal'].drop('MedHouseVal').sort_values(ascending=False))

In [ ]:
# Geographic visualization
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

scatter1 = axes[0].scatter(df['Longitude'], df['Latitude'], c=df['MedHouseVal'],
                           cmap='RdYlGn', s=1, alpha=0.5)
axes[0].set_title('House Prices by Location', fontsize=13)
axes[0].set_xlabel('Longitude')
axes[0].set_ylabel('Latitude')
plt.colorbar(scatter1, ax=axes[0], label='Price ($100k)')

scatter2 = axes[1].scatter(df['Longitude'], df['Latitude'], c=df['MedInc'],
                           cmap='viridis', s=1, alpha=0.5)
axes[1].set_title('Median Income by Location', fontsize=13)
axes[1].set_xlabel('Longitude')
axes[1].set_ylabel('Latitude')
plt.colorbar(scatter2, ax=axes[1], label='Median Income')

plt.tight_layout()
plt.show()

## 3. Feature Engineering

In [ ]:
# Log-transform the target (reduces skewness from capped values)
df['LogPrice'] = np.log1p(df['MedHouseVal'])

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(df['MedHouseVal'], bins=50, color='#e74c3c', edgecolor='white')
axes[0].set_title('Original Price Distribution', fontsize=13)
axes[1].hist(df['LogPrice'], bins=50, color='#2ecc71', edgecolor='white')
axes[1].set_title('Log-Transformed Price Distribution', fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# Prepare features
feature_cols = ['MedInc', 'HouseAge', 'AveRooms', 'AveBedrms', 'Population',
                'AveOccup', 'Latitude', 'Longitude']

X = df[feature_cols].copy()
y = df['LogPrice'].copy()  # Use log-transformed target

# Add polynomial features for top correlated features
poly = PolynomialFeatures(degree=2, include_bias=False, interaction_only=True)
top_features = X[['MedInc', 'AveRooms', 'Latitude', 'Longitude']]
poly_features = poly.fit_transform(top_features)
poly_names = poly.get_feature_names_out(['MedInc', 'AveRooms', 'Latitude', 'Longitude'])

# Keep only interaction terms (skip the original columns already in X)
interaction_cols = [name for name in poly_names if ' ' in name]
interaction_idx = [list(poly_names).index(name) for name in interaction_cols]
X_interactions = pd.DataFrame(poly_features[:, interaction_idx], columns=interaction_cols, index=X.index)

X_full = pd.concat([X, X_interactions], axis=1)
print(f'Original features: {X.shape[1]}')
print(f'With interactions: {X_full.shape[1]}')
print(f'Interaction features: {interaction_cols}')

In [ ]:
# Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X_full, y, test_size=0.2, random_state=42
)

# Scale features for linear models
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f'Train set: {X_train.shape[0]} samples')
print(f'Test set:  {X_test.shape[0]} samples')

## 4. Model Comparison

In [ ]:
models = {
    'Linear Regression': LinearRegression(),
    'Ridge (alpha=1.0)': Ridge(alpha=1.0),
    'Lasso (alpha=0.001)': Lasso(alpha=0.001),
    'Random Forest': RandomForestRegressor(n_estimators=200, max_depth=15, random_state=42, n_jobs=-1),
    'Gradient Boosting': GradientBoostingRegressor(n_estimators=200, max_depth=5, learning_rate=0.1, random_state=42)
}

results = []

for name, model in models.items():
    # Use scaled data for linear models, raw for tree-based
    if 'Forest' in name or 'Boosting' in name:
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)
        cv_scores = cross_val_score(model, X_train, y_train, cv=5, scoring='r2')
    else:
        model.fit(X_train_scaled, y_train)
        y_pred = model.predict(X_test_scaled)
        cv_scores = cross_val_score(model, X_train_scaled, y_train, cv=5, scoring='r2')
    
    # Convert predictions back from log space for RMSE in original units
    y_pred_original = np.expm1(y_pred)
    y_test_original = np.expm1(y_test)
    
    rmse = np.sqrt(mean_squared_error(y_test_original, y_pred_original))
    r2 = r2_score(y_test, y_pred)
    cv_mean = cv_scores.mean()
    
    results.append({
        'Model': name,
        'RMSE ($100k)': round(rmse, 4),
        'R2 (test)': round(r2, 4),
        'R2 (CV mean)': round(cv_mean, 4)
    })
    print(f'{name:25s}  RMSE: {rmse:.4f}  R2: {r2:.4f}  CV R2: {cv_mean:.4f}')

results_df = pd.DataFrame(results).sort_values('R2 (test)', ascending=False)
results_df

## 5. Evaluation: Best Model Analysis

In [ ]:
# Use Gradient Boosting as best model for detailed analysis
best_model = models['Gradient Boosting']
y_pred_best = best_model.predict(X_test)

# Convert back to original scale
y_pred_original = np.expm1(y_pred_best)
y_test_original = np.expm1(y_test)
residuals = y_test_original - y_pred_original

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Predicted vs Actual
axes[0].scatter(y_test_original, y_pred_original, alpha=0.3, s=5, color='#3498db')
max_val = max(y_test_original.max(), y_pred_original.max())
axes[0].plot([0, max_val], [0, max_val], 'r--', linewidth=2, label='Perfect prediction')
axes[0].set_xlabel('Actual Price ($100k)', fontsize=12)
axes[0].set_ylabel('Predicted Price ($100k)', fontsize=12)
axes[0].set_title('Predicted vs Actual', fontsize=13)
axes[0].legend()

# Residual distribution
axes[1].hist(residuals, bins=50, color='#2ecc71', edgecolor='white', alpha=0.8)
axes[1].axvline(0, color='red', linestyle='--', linewidth=2)
axes[1].set_xlabel('Residual ($100k)', fontsize=12)
axes[1].set_ylabel('Count', fontsize=12)
axes[1].set_title(f'Residual Distribution (mean={residuals.mean():.3f})', fontsize=13)

# Residuals vs Predicted
axes[2].scatter(y_pred_original, residuals, alpha=0.3, s=5, color='#e74c3c')
axes[2].axhline(0, color='black', linestyle='--', linewidth=1)
axes[2].set_xlabel('Predicted Price ($100k)', fontsize=12)
axes[2].set_ylabel('Residual ($100k)', fontsize=12)
axes[2].set_title('Residuals vs Predicted', fontsize=13)

plt.tight_layout()
plt.show()

In [ ]:
# Feature importance (Gradient Boosting)
importances = pd.Series(
    best_model.feature_importances_,
    index=X_full.columns
).sort_values(ascending=True)

plt.figure(figsize=(10, 6))
importances.tail(14).plot(kind='barh', color='#3498db')
plt.title('Feature Importance (Gradient Boosting)', fontsize=14)
plt.xlabel('Importance')
plt.tight_layout()
plt.show()

In [ ]:
# Model comparison chart
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

colors = ['#3498db', '#2ecc71', '#e67e22', '#9b59b6', '#e74c3c']

# R2 comparison
axes[0].barh(results_df['Model'], results_df['R2 (test)'], color=colors)
axes[0].set_xlabel('R-squared (Test Set)')
axes[0].set_title('Model Comparison: R-squared', fontsize=13)
axes[0].set_xlim(0, 1)

# RMSE comparison
axes[1].barh(results_df['Model'], results_df['RMSE ($100k)'], color=colors)
axes[1].set_xlabel('RMSE ($100k)')
axes[1].set_title('Model Comparison: RMSE', fontsize=13)

plt.tight_layout()
plt.show()

## Summary

**Key Findings:**
- Median income is by far the strongest predictor of house prices
- Geographic location (latitude/longitude) captures significant price variation
- Log-transforming the target variable improves model performance
- Interaction features add modest improvements for linear models

**Model Comparison:**
- Gradient Boosting achieves the best R-squared and lowest RMSE
- Random Forest is a close second, with less tendency to overfit
- Ridge/Lasso regularization provides marginal benefit over plain linear regression
- All models struggle with the capped target values at $500k

**Residual Analysis:**
- Residuals are approximately normally distributed around zero
- Some heteroscedasticity visible at higher predicted values (capped prices)